In [8]:
import os
import cdsapi
import xarray as xr
import numpy as np
import zipfile


In [9]:
def prepare_graphcast_input(date: str, times: list[str], levels: int = 13):
    assert levels in [13, 37], "Only 13 or 37 pressure levels supported."

    # Set pressure levels
    levels_13 = ['50', '100', '150', '200', '250', '300', '400', '500',
                 '600', '700', '850', '925', '1000']
    levels_37 = [str(l) for l in (
        [1, 2, 3, 5, 7, 10, 20, 30, 50, 70, 100, 125, 150, 175, 200, 225, 250,
         300, 350, 400, 450, 500, 550, 600, 650, 700, 750, 775, 800, 825, 850,
         875, 900, 925, 950, 975, 1000]
    )]

    pressure_levels = levels_13 if levels == 13 else levels_37

    year, month, day = date.split("-")
    time_tag = "-".join(t.replace(":", "") for t in times)
    tag = f"{date}-{levels}lev-{time_tag}"
    folder = f"data/{tag}"
    os.makedirs(folder, exist_ok=True)

    c = cdsapi.Client()

    print("📥 Downloading pressure-level data...")
    c.retrieve(
        'reanalysis-era5-pressure-levels',
        {
            'product_type': 'reanalysis',
            'variable': [
                'temperature', 'u_component_of_wind', 'v_component_of_wind',
                'geopotential', 'vertical_velocity', 'specific_humidity'
            ],
            'pressure_level': pressure_levels,
            'year': year, 'month': month, 'day': day,
            'time': times,
            'format': 'netcdf',
        },
        f"{folder}/era5_pressure.nc"
    )

    print("📥 Downloading surface-level data...")
    surface_path = f"{folder}/era5_surface.zip"
    c.retrieve(
        'reanalysis-era5-single-levels',
        {
            'product_type': 'reanalysis',
            'variable': [
                '2m_temperature', '10m_u_component_of_wind',
                '10m_v_component_of_wind', 'mean_sea_level_pressure',
                'total_precipitation'
            ],
            'year': year, 'month': month, 'day': day,
            'time': times,
            'format': 'netcdf',
        },
        surface_path
    )

    print("🗜️  Extracting surface ZIP...")
    extract_dir = f"{folder}/surface_extracted"
    os.makedirs(extract_dir, exist_ok=True)
    with zipfile.ZipFile(surface_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)

    # Load pressure
    pressure_ds = xr.open_dataset(f"{folder}/era5_pressure.nc", engine="netcdf4")
    pressure_ds = pressure_ds.rename({
        "valid_time": "time", "latitude": "lat", "longitude": "lon", "pressure_level": "level"
    }).expand_dims("batch")

    # Load and merge surface
    instant_ds = xr.open_dataset(f"{extract_dir}/data_stream-oper_stepType-instant.nc", engine="netcdf4")
    accum_ds = xr.open_dataset(f"{extract_dir}/data_stream-oper_stepType-accum.nc", engine="netcdf4")
    surface_ds = xr.merge([instant_ds, accum_ds])
    surface_ds = surface_ds.rename({
        "valid_time": "time", "latitude": "lat", "longitude": "lon"
    }).expand_dims("batch")
    surface_ds["time"] = pressure_ds["time"]

    combined_ds = xr.merge([pressure_ds, surface_ds])
    combined_ds = combined_ds.rename({
        "t": "temperature",
        "u": "u_component_of_wind",
        "v": "v_component_of_wind",
        "z": "geopotential",
        "w": "vertical_velocity",
        "q": "specific_humidity",
        "t2m": "2m_temperature",
        "u10": "10m_u_component_of_wind",
        "v10": "10m_v_component_of_wind",
        "msl": "mean_sea_level_pressure",
        "tp": "total_precipitation_6hr"
    }).drop_vars(["number", "expver"], errors="ignore")

    # Add datetime coordinate
    time_coord = combined_ds["time"]
    batch_size = combined_ds.sizes["batch"]
    datetime_broadcast = xr.DataArray(
        np.broadcast_to(time_coord.values, (batch_size, len(time_coord))),
        dims=("batch", "time")
    )
    combined_ds = combined_ds.assign_coords(datetime=datetime_broadcast)

    # geopotential_at_surface (static from 1000 hPa)
    combined_ds["geopotential_at_surface"] = combined_ds["geopotential"].sel(level=1000).isel(time=0)

    print("📥 Downloading land-sea mask...")
    c.retrieve(
        'reanalysis-era5-single-levels',
        {
            'product_type': 'reanalysis',
            'variable': ['land_sea_mask'],
            'year': year, 'month': month, 'day': day,
            'time': ['00:00'],
            'format': 'netcdf',
        },
        f"{folder}/land_sea_mask.nc"
    )
    lsm_ds = xr.open_dataset(f"{folder}/land_sea_mask.nc")
    lsm_ds = lsm_ds.rename({"valid_time": "time", "latitude": "lat", "longitude": "lon"})
    lsm = lsm_ds["lsm"].isel(time=0).squeeze()
    combined_ds["land_sea_mask"] = lsm

    # Ensure 'geopotential_at_surface' has no time dim
    if "time" in combined_ds["geopotential_at_surface"].dims:
        combined_ds["geopotential_at_surface"] = combined_ds["geopotential_at_surface"].isel(time=0)

    # Save with clear filename
    output_file = f"{folder}/graphcast_ready_input_{tag}.nc"
    combined_ds.to_netcdf(output_file)
    print(f"✅ Saved: {output_file}")


In [11]:
prepare_graphcast_input("2022-02-18", ["06:00", "12:00", "18:00"], levels=37)


2025-05-06 15:01:44,275 INFO [2024-09-26T00:00:00] Watch our [Forum](https://forum.ecmwf.int/) for Announcements, news and other discussed topics.
2025-05-06 15:01:44,276 WARNING [2024-06-16T00:00:00] CDS API syntax is changed and some keys or parameter names may have also changed. To avoid requests failing, please use the "Show API request code" tool on the dataset Download Form to check you are using the correct syntax for your API request.


📥 Downloading pressure-level data...


2025-05-06 15:01:44,548 INFO Request ID is 015c7392-9365-40bc-b81f-80f148256e4a
2025-05-06 15:01:44,714 INFO status has been updated to accepted
2025-05-06 15:02:06,624 INFO status has been updated to running
2025-05-06 15:10:09,067 INFO status has been updated to successful


dc679f9edd89cb48735dbeec3713331.nc:   0%|          | 0.00/1.09G [00:00<?, ?B/s]

📥 Downloading surface-level data...


...

9c0ef613646ab7da212a05c645ea8d19.zip:   0%|          | 0.00/22.3M [00:00<?, ?B/s]

🗜️  Extracting surface ZIP...
📥 Downloading land-sea mask...


2025-05-06 15:11:57,458 INFO Request ID is f4c0004e-3e27-41f3-8b0d-10a3a2ec6eac
2025-05-06 15:11:57,529 INFO status has been updated to accepted
2025-05-06 15:12:30,655 INFO status has been updated to running
2025-05-06 15:12:48,898 INFO status has been updated to successful


576bae73267b88f816825d2f9ab4b61.nc:   0%|          | 0.00/762k [00:00<?, ?B/s]

✅ Saved: data/2022-02-18-37lev-0600-1200-1800/graphcast_ready_input_2022-02-18-37lev-0600-1200-1800.nc
